In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

In [ ]:
# Import data
niv = '3'

# HICP All Items
colHicp = "INX"

# Unchained other items 
df_inflation = pd.read_parquet(f"input/chainedItemsLvl{niv}.parquet")
df_inflation 

ICP_ITEM,011100,011200,011300,011400,011500,011600,011700,011800,011900,012100,...,121400,121900,122200,122900,131100,131200,131300,132100,132200,132900
TIME_PERIOD,,,,,,,,,,,,,,,,,,,,,
1997-01-01,1.783415,3.415818,0.823227,1.792659,0.485575,0.969382,1.312325,0.800837,0.564981,0.180342,...,0.393752,0.000695,NaN,NaN,0.061747,1.163640,0.971900,0.540958,0.000000,0.357667
1997-02-01,1.786212,3.408498,0.812145,1.795017,0.477699,0.985655,1.260876,0.803046,0.564527,0.180451,...,0.394708,0.000695,NaN,NaN,0.061864,1.167479,0.974342,0.540958,0.000000,0.358462
1997-03-01,1.787260,3.403840,0.809704,1.794343,0.470260,0.983169,1.227187,0.803782,0.564073,0.180488,...,0.395149,0.000698,NaN,NaN,0.061934,1.169559,0.976221,0.541888,0.000000,0.359069
1997-04-01,1.787260,3.405836,0.809704,1.791649,0.465447,0.998086,1.248602,0.803340,0.564437,0.180451,...,0.395443,0.000697,NaN,NaN,0.061968,1.169879,0.977724,0.541733,0.000000,0.359209
1997-05-01,1.787959,3.447760,0.812521,1.789291,0.461728,1.009839,1.270800,0.803782,0.564255,0.180342,...,0.395369,0.000698,NaN,NaN,0.062065,1.170998,0.979979,0.541888,0.000000,0.359443
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-01,2.721027,3.443154,0.965502,2.280134,0.509281,1.461361,2.042656,1.107335,1.291620,0.232339,...,0.940815,0.228288,1.033792,0.100710,0.063441,1.669056,1.433497,0.590688,0.001142,0.497751
2026-05-01,2.723192,3.446165,0.967557,2.275571,0.506951,1.515442,1.955540,1.104511,1.290977,0.232804,...,0.948797,0.228180,1.034402,0.102015,0.064003,1.667724,1.437253,0.595008,0.001168,0.497702
2026-06-01,2.725086,3.442150,0.966623,2.270780,0.505814,1.497607,1.915611,1.107227,1.293549,0.231642,...,0.936510,0.226223,1.034504,0.101228,0.063902,1.673215,1.440035,0.594850,0.001184,0.496327


In [ ]:
def replicate_ismi(df_inflation, df_weights, rolling_window=120, k=3):
    """Replication de l'indice ISMI (Lansing & Shapiro, 2026).

    df_inflation : DataFrame (Dates x Catégories) - Taux d'inflation mensuels
    df_weights   : DataFrame (Dates x Catégories) - Poids de dépenses (PCE)
    """
    categories = df_inflation.columns
    dates = df_inflation.index[rolling_window:]

    ismi_series = []
    pos_momentum_series = []
    neg_momentum_series = []

    # Stockage temporaire des chocs (résidus de l'AR(1))
    shocks_dict = {cat: pd.Series(index=df_inflation.index, dtype=float) for cat in categories}

    print("Étape 1 : Estimation des chocs via AR(1) glissant...")
    for t_idx in range(rolling_window, len(df_inflation)):
        current_date = df_inflation.index[t_idx]

        for cat in categories:
            # Fenêtre glissante de 120 mois (Lansing & Shapiro, 2026)
            window_data = df_inflation[cat].iloc[t_idx - rolling_window : t_idx + 1]

            y = window_data.iloc[1:]  # t
            x = window_data.iloc[:-1]  # t-1
            x = sm.add_constant(x)

            # Régression linéaire pour obtenir le choc du mois courant
            model = sm.OLS(y, x).fit()
            # Le dernier résidu correspond au choc du mois t
            shocks_dict[cat].at[current_date] = model.resid.iloc[-1]

    df_shocks = pd.DataFrame(shocks_dict)

    print("Étape 2 : Calcul du momentum et agrégation pondérée...")
    for current_date in dates:
        # Récupérer l'historique récent des chocs pour la règle des k mois consécutifs
        t_pos = df_shocks.index.get_loc(current_date)
        recent_shocks = df_shocks.iloc[t_pos - k + 1 : t_pos + 1]

        # Détection des chocs directionnels consécutifs (ex: 3 mois positifs)
        pos_signal = (recent_shocks > 0).all(axis=0).astype(int)
        neg_signal = (recent_shocks < 0).all(axis=0).astype(int)

        # # Extraction des poids pour le mois en cours
        # w = df_weights.loc[current_date]
        # w_normalized = w / w.sum()

        # # Calcul des parts pondérées de momentum positive et négative
        # share_pos = np.dot(pos_signal, w_normalized)
        # share_neg = np.dot(neg_signal, w_normalized)

        # # ISMI = part positive - part négative
        # ismi = share_pos - share_neg
        ismi = pos_signal - neg_signal

        ismi_series.append(ismi)
        pos_momentum_series.append(share_pos)
        neg_momentum_series.append(share_neg)

    results = pd.DataFrame(
        {
            "ISMI": ismi_series,
            "Positive_Momentum": pos_momentum_series,
            "Negative_Momentum": neg_momentum_series,
        },
        index=dates,
    )

    return results

In [ ]:
categories = df_inflation.columns
dates = df_inflation.index[120:]

ismi_series = []
pos_momentum_series = []
neg_momentum_series = []

# Stockage temporaire des chocs (résidus de l'AR(1))
shocks_dict = {cat: pd.Series(index=df_inflation.index, dtype=float) for cat in categories}

print("Étape 1 : Estimation des chocs via AR(1) glissant...")
for t_idx in range(120, len(df_inflation)):
    current_date = df_inflation.index[t_idx]

    for cat in categories:
        # Fenêtre glissante de 120 mois (Lansing & Shapiro, 2026)
        window_data = df_inflation[cat].iloc[t_idx - rolling_window : t_idx + 1]

        y = window_data.iloc[1:]  # t
        x = window_data.iloc[:-1]  # t-1
        x = sm.add_constant(x)

        # Régression linéaire pour obtenir le choc du mois courant
        model = sm.OLS(y, x).fit()
        # Le dernier résidu correspond au choc du mois t
        shocks_dict[cat].at[current_date] = model.resid.iloc[-1]

df_shocks = pd.DataFrame(shocks_dict)